# CardioScore Validation 03 — Locked External Test

This is the headline external test. No tuning is performed from the uploaded validation results.

In [ ]:

import sys, subprocess, json, hashlib, zipfile, tarfile
from pathlib import Path
PIN = "869150cd5fb5ccf155fb066258404bd4df163ade"
REPO = "Virelion-Biotech/Virelion-CardioScore"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"git+https://github.com/{REPO}.git@{PIN}"], check=True)
print("Installed pinned CardioScore:", PIN)


In [ ]:

from google.colab import files
up = files.upload()
upload_dir = Path("/content/cardioscore_validation/locked_input")
upload_dir.mkdir(parents=True, exist_ok=True)
for name, data in up.items(): (upload_dir/name).write_bytes(data)
required = ["canonical_features.csv","reference.csv","source_receipt.json","derivation_manifest.json"]
missing = [x for x in required if not (upload_dir/x).exists()]
assert not missing, f"Missing locked-validation file(s): {missing}"
features = pd.read_csv(upload_dir/"canonical_features.csv")
reference = pd.read_csv(upload_dir/"reference.csv")
receipt = json.loads((upload_dir/"source_receipt.json").read_text())
derivation = json.loads((upload_dir/"derivation_manifest.json").read_text())


In [ ]:

from virelion_cardioscore.analysis.pipeline import CardioScorePipeline
from virelion_cardioscore.validation.manifest import validate_feature_schema, validate_reference_schema, validate_vehicle_structure
from virelion_cardioscore.validation.metrics import locked_metrics, stratified_failures

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for b in iter(lambda:f.read(1024*1024),b""): h.update(b)
    return h.hexdigest()

assert receipt.get("sha256"), "Source receipt must contain a source SHA-256."
assert derivation.get("verified_rebuild") is True, "Headline external validation requires verified_rebuild=true."
assert derivation.get("source_sha256") == receipt["sha256"], "Derivation source hash does not match immutable receipt."
assert derivation.get("canonical_features_sha256") == sha256_file(upload_dir/"canonical_features.csv")
assert derivation.get("reference_sha256") == sha256_file(upload_dir/"reference.csv")
assert derivation.get("config_sha256"), "Record the exact config hash used to derive the locked table."
validate_feature_schema(features)
validate_reference_schema(reference)
if bool(__import__("yaml").safe_load(Path(__import__("virelion_cardioscore").__file__).read_text(errors="ignore") if False else "{}").get("scoring",{}).get("normalize_by_vehicle",True)):
    validate_vehicle_structure(features)

for c in features.columns:
    assert "reference_risk" not in c.lower() and "risk_class" not in c.lower(), f"Leakage-prone label column in features: {c}"

p = CardioScorePipeline.from_defaults()
frozen = json.loads(json.dumps(p.config, default=str))
result = p.run(features)
obs = result.summary_table[["compound","risk_class","cardioscore"]].rename(columns={"risk_class":"observed_risk"})
joined = reference.merge(obs, on="compound", how="inner", validate="one_to_one")
assert len(joined) == len(reference), "Not every reference compound is scoreable; report exclusions separately before any headline claim."
metrics = locked_metrics(joined["reference_risk"], joined["observed_risk"])
failures = stratified_failures(joined, strata=("compound",))


In [ ]:

out = Path("/content/cardioscore_validation/results")
out.mkdir(parents=True, exist_ok=True)
payload = {
    "package_commit": PIN,
    "locked": True,
    "source_id": receipt.get("source_id"),
    "source_sha256": receipt.get("sha256"),
    "derivation_manifest": derivation,
    "metrics": metrics.to_dict(),
    "compound_scores": joined.to_dict(orient="records"),
    "failures": failures.to_dict(orient="records"),
    "excluded_or_unscoreable_compounds": sorted(set(reference["compound"]) - set(joined["compound"]))
}
(out/"locked_external_validation.json").write_text(json.dumps(payload, indent=2, allow_nan=False)+"\n")
joined.to_csv(out/"locked_compound_validation.csv", index=False)
failures.to_csv(out/"locked_failures_by_compound.csv", index=False)
print(json.dumps(metrics.to_dict(), indent=2))


### Required lineage
`source_receipt.json` → source SHA; `derivation_manifest.json` → source SHA, feature/reference hashes, adapter commit, config hash, and `verified_rebuild: true`. Without that chain, do not report the validation result as a locked external result.